<a href="https://colab.research.google.com/github/ade071212/Tugas-Modul-ML/blob/main/Tugas_Modul_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Latihan 4 – Machine Learning
## Pipeline Preprocessing dan Seleksi Fitur: Prediksi Kelulusan Tepat Waktu

## 1. Import Library dan Membuat Data

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn import set_config

set_config(display="text")

In [ ]:
# Dataset sintetis 800 mahasiswa (ganti dengan pd.read_csv(...) bila memakai data asli)
gen = np.random.default_rng(99)
n = 800

ipk        = np.clip(gen.normal(3.1, 0.45, n), 1.5, 4.0)
jam_belajar = np.clip(gen.normal(14, 6, n), 0, 40)
kehadiran  = np.clip(gen.normal(82, 10, n), 40, 100)
umur       = gen.integers(19, 26, n)                      # sengaja dibuat kurang berpengaruh
jurusan    = gen.choice(["Teknik", "Ekonomi", "Hukum", "Sains"], n, p=[0.35, 0.30, 0.15, 0.20])
kerja      = gen.choice(["Tidak", "Paruh waktu", "Penuh waktu"], n, p=[0.6, 0.3, 0.1])

skor = (2.2 * (ipk - 3.1) + 0.06 * (jam_belajar - 14) + 0.04 * (kehadiran - 82)
        - 0.9 * (kerja == "Penuh waktu") - 0.3 * (kerja == "Paruh waktu")
        + 0.4 * (jurusan == "Sains") + gen.normal(0, 0.7, n) + 0.3)
lulus = (gen.random(n) < 1 / (1 + np.exp(-skor))).astype(int)

mhs = pd.DataFrame({"ipk": ipk.round(2), "jam_belajar": jam_belajar.round(1),
                    "kehadiran": kehadiran.round(1), "umur": umur,
                    "jurusan": jurusan, "status_kerja": kerja, "lulus_tepat": lulus})

# Sisipkan nilai kosong secara acak
for kol, frac in [("ipk", 0.07), ("jam_belajar", 0.10), ("jurusan", 0.05), ("status_kerja", 0.06)]:
    mhs.loc[gen.choice(n, int(frac * n), replace=False), kol] = np.nan

print("Ukuran data:", mhs.shape)
mhs.head()

Ukuran data: (800, 7)


,ipk,jam_belajar,kehadiran,umur,jurusan,status_kerja,lulus_tepat
0,3.14,15.9,87.8,20,Teknik,Paruh waktu,1
1,2.89,18.3,63.7,19,Teknik,Tidak,0
2,3.12,3.9,90.8,20,Hukum,Tidak,0
3,3.41,NaN,79.5,20,Ekonomi,Tidak,1
4,NaN,9.0,89.6,20,Ekonomi,Tidak,0


In [ ]:
print("Nilai kosong per kolom:")
print(mhs.isna().sum())
print("\nProporsi kelas target:")
print(mhs["lulus_tepat"].value_counts(normalize=True).round(3))

Nilai kosong per kolom:
ipk             56
jam_belajar     80
kehadiran        0
umur             0
jurusan         40
status_kerja    48
lulus_tepat      0
dtype: int64

Proporsi kelas target:
lulus_tepat
1    0.556
0    0.444
Name: proportion, dtype: float64


## 2. Split Data Terstratifikasi

In [ ]:
fitur = mhs.drop(columns="lulus_tepat")
target = mhs["lulus_tepat"]

X_tr, X_te, y_tr, y_te = train_test_split(
    fitur, target, test_size=0.25, stratify=target, random_state=5)

print("Latih:", X_tr.shape, "| Uji:", X_te.shape)
print("Proporsi lulus tepat (latih):", round(y_tr.mean(), 4))
print("Proporsi lulus tepat (uji)  :", round(y_te.mean(), 4))

Latih: (600, 6) | Uji: (200, 6)
Proporsi lulus tepat (latih): 0.5567
Proporsi lulus tepat (uji)  : 0.555


## 3. Menentukan Jenis Kolom

In [ ]:
kol_angka = fitur.select_dtypes(include="number").columns.tolist()
kol_teks = fitur.select_dtypes(exclude="number").columns.tolist()

print("Kolom numerik    :", kol_angka)
print("Kolom kategorikal:", kol_teks)

Kolom numerik    : ['ipk', 'jam_belajar', 'kehadiran', 'umur']
Kolom kategorikal: ['jurusan', 'status_kerja']


## 4. Membangun ColumnTransformer dan Pipeline

In [ ]:
alur_angka = Pipeline([
    ("isi", SimpleImputer(strategy="mean")),
    ("skala", RobustScaler())])

alur_teks = Pipeline([
    ("isi", SimpleImputer(strategy="constant", fill_value="Tidak diketahui")),
    ("ohe", OneHotEncoder(handle_unknown="ignore"))])

praproses = ColumnTransformer([
    ("angka", alur_angka, kol_angka),
    ("teks", alur_teks, kol_teks)])

model_penuh = Pipeline([
    ("praproses", praproses),
    ("clf", LogisticRegression(max_iter=1000))])

model_penuh.fit(X_tr, y_tr)
print("Akurasi uji (semua fitur):", round(accuracy_score(y_te, model_penuh.predict(X_te)), 4))

Akurasi uji (semua fitur): 0.63


In [ ]:
nama_kolom = model_penuh.named_steps["praproses"].get_feature_names_out()
print("Fitur setelah praproses:")
for nm in nama_kolom:
    print(" *", nm)

isi_angka = model_penuh.named_steps["praproses"].named_transformers_["angka"].named_steps["isi"]
print("\nRata-rata yang dipelajari dari data latih:")
print(dict(zip(kol_angka, isi_angka.statistics_.round(3))))

Fitur setelah praproses:
 * angka__ipk
 * angka__jam_belajar
 * angka__kehadiran
 * angka__umur
 * teks__jurusan_Ekonomi
 * teks__jurusan_Hukum
 * teks__jurusan_Sains
 * teks__jurusan_Teknik
 * teks__jurusan_Tidak diketahui
 * teks__status_kerja_Paruh waktu
 * teks__status_kerja_Penuh waktu
 * teks__status_kerja_Tidak
 * teks__status_kerja_Tidak diketahui

Rata-rata yang dipelajari dari data latih:
{'ipk': np.float64(3.091), 'jam_belajar': np.float64(14.054), 'kehadiran': np.float64(81.877), 'umur': np.float64(22.018)}


## 5. Analisis Fitur sebelum Seleksi

In [ ]:
# Chi-square antara fitur kategorikal dan target (data latih saja)
baris = []
for kol in kol_teks:
    ct = pd.crosstab(X_tr[kol].fillna("Tidak diketahui"), y_tr)
    chi2, p, dof, _ = chi2_contingency(ct)
    baris.append({"fitur": kol, "chi2": round(chi2, 3), "dof": dof, "p_value": round(p, 4)})

uji_chi = pd.DataFrame(baris)
uji_chi["kesimpulan"] = np.where(uji_chi["p_value"] < 0.05, "berhubungan", "tidak signifikan")
uji_chi

,fitur,chi2,dof,p_value,kesimpulan
0,jurusan,6.233,4,0.1824,tidak signifikan
1,status_kerja,9.129,3,0.0276,berhubungan


In [ ]:
# Korelasi antar fitur numerik untuk melihat redundansi
korelasi = X_tr[kol_angka].corr().round(2)
korelasi

,ipk,jam_belajar,kehadiran,umur
ipk,1.00,0.07,-0.04,0.04
jam_belajar,0.07,1.00,0.04,-0.06
kehadiran,-0.04,0.04,1.00,-0.01
umur,0.04,-0.06,-0.01,1.00


In [ ]:
# Skor Mutual Information untuk setiap fitur hasil praproses
Xp = model_penuh.named_steps["praproses"].transform(X_tr)
mi = mutual_info_classif(Xp, y_tr, random_state=5)

peringkat = (pd.DataFrame({"fitur": nama_kolom, "skor_MI": mi.round(4)})
             .sort_values("skor_MI", ascending=False).reset_index(drop=True))
peringkat

,fitur,skor_MI
0,angka__ipk,0.0542
1,angka__kehadiran,0.0195
2,teks__jurusan_Ekonomi,0.0180
3,teks__status_kerja_Tidak,0.0161
4,teks__status_kerja_Penuh waktu,0.0151
5,teks__jurusan_Sains,0.0136
6,teks__status_kerja_Tidak diketahui,0.0082
7,angka__jam_belajar,0.0015
8,angka__umur,0.0000
9,teks__jurusan_Tidak diketahui,0.0000


## 6. Pipeline dengan Seleksi Fitur (SelectKBest, k=5)

In [ ]:
K = 5
model_seleksi = Pipeline([
    ("praproses", praproses),
    ("pilih", SelectKBest(score_func=lambda X, y: mutual_info_classif(X, y, random_state=5), k=K)),
    ("clf", LogisticRegression(max_iter=1000))])

model_seleksi.fit(X_tr, y_tr)

aktif = model_seleksi.named_steps["pilih"].get_support()
print(f"Fitur DIPAKAI (k={K}):")
for f in nama_kolom[aktif]: print("  [+]", f)
print("\nFitur DIBUANG:")
for f in nama_kolom[~aktif]: print("  [-]", f)

Fitur DIPAKAI (k=5):
  [+] angka__ipk
  [+] angka__kehadiran
  [+] teks__jurusan_Ekonomi
  [+] teks__status_kerja_Penuh waktu
  [+] teks__status_kerja_Tidak

Fitur DIBUANG:
  [-] angka__jam_belajar
  [-] angka__umur
  [-] teks__jurusan_Hukum
  [-] teks__jurusan_Sains
  [-] teks__jurusan_Teknik
  [-] teks__jurusan_Tidak diketahui
  [-] teks__status_kerja_Paruh waktu
  [-] teks__status_kerja_Tidak diketahui


## 7. Evaluasi dan Perbandingan

In [ ]:
validasi = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=5)

def ringkas_skor(judul, mdl, n_fitur):
    pred = mdl.predict(X_te)
    cv = cross_validate(mdl, X_tr, y_tr, cv=validasi, scoring=["accuracy", "f1"])
    return {"Skenario": judul, "Jml fitur": n_fitur,
            "Acc uji": round(accuracy_score(y_te, pred), 4),
            "F1 uji": round(f1_score(y_te, pred), 4),
            "Acc CV": round(cv["test_accuracy"].mean(), 4),
            "F1 CV": round(cv["test_f1"].mean(), 4)}

perbandingan = pd.DataFrame([
    ringkas_skor("Semua fitur", model_penuh, len(nama_kolom)),
    ringkas_skor(f"SelectKBest (k={K})", model_seleksi, K)])
perbandingan

,Skenario,Jml fitur,Acc uji,F1 uji,Acc CV,F1 CV
0,Semua fitur,13,0.630,0.6838,0.6722,0.7153
1,SelectKBest (k=5),5,0.595,0.6667,0.6733,0.7195


In [ ]:
for judul, mdl in [("SEMUA FITUR", model_penuh), (f"FITUR TERPILIH (k={K})", model_seleksi)]:
    print(f"--- {judul} ---")
    print(classification_report(y_te, mdl.predict(X_te), target_names=["Terlambat", "Tepat waktu"]))

--- SEMUA FITUR ---
              precision    recall  f1-score   support

   Terlambat       0.60      0.52      0.55        89
 Tepat waktu       0.65      0.72      0.68       111

    accuracy                           0.63       200
   macro avg       0.62      0.62      0.62       200
weighted avg       0.63      0.63      0.63       200

--- FITUR TERPILIH (k=5) ---
              precision    recall  f1-score   support

   Terlambat       0.56      0.43      0.48        89
 Tepat waktu       0.61      0.73      0.67       111

    accuracy                           0.59       200
   macro avg       0.59      0.58      0.58       200
weighted avg       0.59      0.59      0.59       200



## 8. Ringkasan Akhir

In [ ]:
print("PIPELINE AKHIR:")
print(model_seleksi)
print("\nFITUR YANG DIGUNAKAN MODEL:")
for no, f in enumerate(nama_kolom[aktif], 1):
    print(f"  {no}. {f}")
print("\nTABEL PERBANDINGAN:")
print(perbandingan.to_string(index=False))

PIPELINE AKHIR:
Pipeline(steps=[('praproses',
                 ColumnTransformer(transformers=[('angka',
                                                  Pipeline(steps=[('isi',
                                                                   SimpleImputer()),
                                                                  ('skala',
                                                                   RobustScaler())]),
                                                  ['ipk', 'jam_belajar',
                                                   'kehadiran', 'umur']),
                                                 ('teks',
                                                  Pipeline(steps=[('isi',
                                                                   SimpleImputer(fill_value='Tidak '
                                                                                            'diketahui',
                                                                                 strategy

### Kesimpulan

In [ ]:
a, b = perbandingan.iloc[0], perbandingan.iloc[1]
selisih_f1 = b["F1 CV"] - a["F1 CV"]
lebih_baik = "lebih baik" if selisih_f1 > 0.005 else ("lebih buruk" if selisih_f1 < -0.005 else "setara")

print("KESIMPULAN")
print("=" * 60)
print("1. Seluruh imputasi, penskalaan, dan encoding berada di dalam Pipeline,")
print("   sehingga statistik hanya dipelajari dari data latih (tanpa kebocoran data).")
print(f"2. Seleksi fitur mengurangi fitur dari {int(a['Jml fitur'])} menjadi {int(b['Jml fitur'])}.")
print(f"   Fitur terbuang: {', '.join(nama_kolom[~aktif])}.")
print(f"3. F1 validasi silang: semua fitur {a['F1 CV']:.3f} vs terpilih {b['F1 CV']:.3f}")
print(f"   (selisih {selisih_f1:+.3f}); model terseleksi {lebih_baik} dibanding model penuh.")
print(f"   Pada data uji: F1 {a['F1 uji']:.3f} vs {b['F1 uji']:.3f}.")
print("4. Model dengan fitur lebih sedikit lebih sederhana dan lebih mudah dijelaskan,")
print("   selama performanya tidak turun berarti.")

KESIMPULAN
1. Seluruh imputasi, penskalaan, dan encoding berada di dalam Pipeline,
   sehingga statistik hanya dipelajari dari data latih (tanpa kebocoran data).
2. Seleksi fitur mengurangi fitur dari 13 menjadi 5.
   Fitur terbuang: angka__jam_belajar, angka__umur, teks__jurusan_Hukum, teks__jurusan_Sains, teks__jurusan_Teknik, teks__jurusan_Tidak diketahui, teks__status_kerja_Paruh waktu, teks__status_kerja_Tidak diketahui.
3. F1 validasi silang: semua fitur 0.715 vs terpilih 0.720
   (selisih +0.004); model terseleksi setara dibanding model penuh.
   Pada data uji: F1 0.684 vs 0.667.
4. Model dengan fitur lebih sedikit lebih sederhana dan lebih mudah dijelaskan,
   selama performanya tidak turun berarti.
